In [2]:
# hsf_dsp.py
#
# Digital Signal Processing module for the Harmonic Spectral Filter (HSF).
# Handles FFT, phase extraction, and phase error computation.

import numpy as np

def compute_fft(signal_window):
    """
    Computes the Fast Fourier Transform (FFT) of a given signal window.

    Args:
        signal_window (np.ndarray): A 1D numpy array representing the signal window.

    Returns:
        np.ndarray: The complex-valued FFT result.
    """
    if len(signal_window) == 0:
        return np.array()
    return np.fft.fft(signal_window)

def get_phase_at_frequency(fft_result, target_freq, sampling_rate):
    """
    Extracts the phase angle for a specific target frequency from an FFT result.

    Args:
        fft_result (np.ndarray): The complex-valued result of an FFT.
        target_freq (float): The target frequency in Hz (e.g., 40 Hz).
        sampling_rate (float): The sampling rate of the original signal in Hz.

    Returns:
        float: The phase angle in radians for the target frequency.
    """
    if len(fft_result) == 0:
        return 0.0
        
    fft_len = len(fft_result)
    freq_resolution = sampling_rate / fft_len
    
    # Find the closest FFT bin index for the target frequency
    target_bin = int(round(target_freq / freq_resolution))
    
    if target_bin >= fft_len:
        raise ValueError("Target frequency is outside the range of the FFT.")

    # Get the complex value at the target bin
    complex_value = fft_result[target_bin]
    
    # Calculate the phase angle
    phase_angle = np.angle(complex_value)
    
    return phase_angle

def calculate_phase_error(current_phase, prev_phase, target_freq, time_step):
    """
    Calculates the phase error based on the PSREQ loop's reflective step.
    This function predicts the expected phase and computes the deviation.

    Args:
        current_phase (float): The current phase angle in radians.
        prev_phase (float): The phase angle from the previous time step in radians.
        target_freq (float): The target frequency in Hz.
        time_step (float): The time elapsed since the last measurement in seconds.

    Returns:
        float: The wrapped phase error (Δφ) in radians, in the range [-π, π].
    """
    # Predict the expected phase based on the previous phase and frequency
    # φ_k* = φ_k(t-T) + ω_k * T
    angular_freq = 2 * np.pi * target_freq
    predicted_phase = prev_phase + angular_freq * time_step
    
    # Calculate the raw phase error
    error = current_phase - predicted_phase
    
    # Wrap the error to the interval [-π, π] for the PID controller
    wrapped_error = (error + np.pi) % (2 * np.pi) - np.pi
    
    return wrapped_error

# --- Unit Tests for hsf_dsp.py ---

import unittest

class TestHsfDsp(unittest.TestCase):

    def setUp(self):
        self.SAMPLING_RATE = 200  # Hz
        self.FFT_LEN = 64
        self.TIME_STEP = self.FFT_LEN / (2 * self.SAMPLING_RATE) # 50% overlap
        self.TARGET_FREQ = 40  # Hz
        self.t = np.arange(self.FFT_LEN) / self.SAMPLING_RATE

    def test_phase_extraction(self):
        # Create a pure 40 Hz sine wave with a known phase shift of π/4
        phase_shift = np.pi / 4
        signal = np.sin(2 * np.pi * self.TARGET_FREQ * self.t + phase_shift)
        
        fft_result = compute_fft(signal)
        extracted_phase = get_phase_at_frequency(fft_result, self.TARGET_FREQ, self.SAMPLING_RATE)
        
        # The phase of a sin(ωt + φ) is φ - π/2
        expected_phase = phase_shift - np.pi / 2
        self.assertAlmostEqual(extracted_phase, expected_phase, places=5)

    def test_phase_error_calculation(self):
        # Test case: current phase matches predicted phase perfectly
        current_phase = 0.5
        prev_phase = 0.2
        target_freq = 40
        time_step = (current_phase - prev_phase) / (2 * np.pi * target_freq)
        error = calculate_phase_error(current_phase, prev_phase, target_freq, time_step)
        self.assertAlmostEqual(error, 0.0, places=5)
        
        # Test wrapping
        error_positive_wrap = calculate_phase_error(0.1, 3.5, 1, 1) # Raw error > π
        self.assertTrue(-np.pi <= error_positive_wrap <= np.pi)
        
        error_negative_wrap = calculate_phase_error(0.1, -3.5, 1, -1) # Raw error < -π
        self.assertTrue(-np.pi <= error_negative_wrap <= np.pi)

if __name__ == '__main__':
    unittest.main(argv=['first-arg-is-ignored'], exit=False)

.F..
FAIL: test_phase_extraction (__main__.TestHsfDsp.test_phase_extraction)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "C:\Users\Developer\AppData\Local\Temp\ipykernel_8228\2508367415.py", line 104, in test_phase_extraction
    self.assertAlmostEqual(extracted_phase, expected_phase, places=5)
AssertionError: -1.3956244996345053 != -0.7853981633974483 within 5 places (0.610226336237057 difference)

----------------------------------------------------------------------
Ran 4 tests in 0.001s

FAILED (failures=1)


In [4]:
# hsf_pid.py
#
# Proportional-Integral-Derivative (PID) controller module for the HSF.
# Implements the Samson v2 feedback control law to minimize phase error.

class SamsonV2_PID:
    """
    A PID controller class that implements the Samson v2 feedback law.
    It calculates a control output to correct for phase error (Δφ).
    """
    def __init__(self, Kp, Ki, Kd, setpoint=0):
        """
        Initializes the PID controller.

        Args:
            Kp (float): Proportional gain.
            Ki (float): Integral gain.
            Kd (float): Derivative gain.
            setpoint (float, optional): The target value for the error. Defaults to 0.
        """
        self.Kp = Kp
        self.Ki = Ki
        self.Kd = Kd
        self.setpoint = setpoint
        
        self.integral_term = 0.0
        self.prev_error = 0.0

    def compute(self, phase_error, dt):
        """
        Computes the control output signal based on the phase error.

        Args:
            phase_error (float): The current phase error (Δφ).
            dt (float): The time delta since the last computation.

        Returns:
            float: The control output signal (u_k).
        """
        if dt <= 0:
            return 0.0

        # Proportional term
        p_term = self.Kp * phase_error
        
        # Integral term
        self.integral_term += phase_error * dt
        i_term = self.Ki * self.integral_term
        
        # Derivative term
        derivative = (phase_error - self.prev_error) / dt
        d_term = self.Kd * derivative
        
        # Update previous error for the next iteration
        self.prev_error = phase_error
        
        # The control law output is the sum of the terms
        control_output = p_term + i_term + d_term
        
        return control_output

    def reset(self):
        """Resets the integral and derivative history of the controller."""
        self.integral_term = 0.0
        self.prev_error = 0.0

# --- Unit Tests for hsf_pid.py ---

import unittest

class TestHsfPid(unittest.TestCase):

    def test_pid_components(self):
        # Test with only a P term
        pid = SamsonV2_PID(Kp=10, Ki=0, Kd=0)
        self.assertEqual(pid.compute(phase_error=5, dt=1), 50)

        # Test with only an I term
        pid = SamsonV2_PID(Kp=0, Ki=10, Kd=0)
        pid.compute(phase_error=5, dt=1)
        self.assertEqual(pid.integral_term, 5)
        self.assertEqual(pid.compute(phase_error=2, dt=1), 70) # 10 * (5+2)

        # Test with only a D term
        pid = SamsonV2_PID(Kp=0, Ki=0, Kd=10)
        pid.prev_error = 1 # Set previous error for derivative calculation
        self.assertEqual(pid.compute(phase_error=5, dt=1), 40) # 10 * (5-1)/1

    def test_pid_reset(self):
        pid = SamsonV2_PID(Kp=1, Ki=1, Kd=1)
        pid.compute(phase_error=5, dt=1)
        self.assertNotEqual(pid.integral_term, 0)
        self.assertNotEqual(pid.prev_error, 0)
        
        pid.reset()
        self.assertEqual(pid.integral_term, 0)
        self.assertEqual(pid.prev_error, 0)

if __name__ == '__main__':
    unittest.main(argv=['first-arg-is-ignored'], exit=False)

.F..
FAIL: test_phase_extraction (__main__.TestHsfDsp.test_phase_extraction)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "C:\Users\Developer\AppData\Local\Temp\ipykernel_8228\2508367415.py", line 104, in test_phase_extraction
    self.assertAlmostEqual(extracted_phase, expected_phase, places=5)
AssertionError: -1.3956244996345053 != -0.7853981633974483 within 5 places (0.610226336237057 difference)

----------------------------------------------------------------------
Ran 4 tests in 0.001s

FAILED (failures=1)


In [9]:
import math

def nexus_byte_recursive_v1(header=(1, 4)):
    """
    A tail-recursive implementation of the nexus_byte algorithm, modeled as a
    finite state machine.

    Args:
        header (tuple): The initial two-element tuple to start the process.

    Returns:
        list: The final generated list of numbers.
    """
    # --- Private Helper Functions for Each Transformation (Pure Functions) ---

    def _expand_universe(current_stack):
        # Expands the stack based on the difference of the first two elements.
        diff = abs(current_stack - current_stack)
        # Use math.log2 for a robust way to get bit length, or bin() for simplicity.
        # len(bin(diff)[2:]) is a common way to get bit length.
        bin_len = len(bin(diff)[2:]) if diff > 0 else 1
        
        # Create a new list instead of mutating in-place.
        new_stack = list(current_stack)
        new_stack.extend([bin_len] * bin_len)
        return new_stack

    def _add_z(current_stack):
        # Calculates 'z' and inserts it at the current end of the expanded stack.
        # Note: In the original, this overwrites the last element from expansion.
        z = current_stack + current_stack
        
        new_stack = list(current_stack)
        # The pointer in the original pointed to the last element.
        new_stack[-1] = z
        return new_stack

    def _stabilize_bit3(current_stack):
        # Calculates a difference and sets it at index 2.
        z = current_stack[-1] # 'z' is the value just added
        present = current_stack
        diff_z = z - present
        
        new_stack = list(current_stack)
        new_stack = diff_z
        return new_stack

    def _add_y(current_stack):
        # Appends 'y' to the stack.
        z = current_stack[-1] # 'z' is at the end before this operation
        present = current_stack
        y = z + present
        
        new_stack = list(current_stack)
        new_stack.append(y)
        return new_stack

    def _add_x(current_stack):
        # Appends the number of initial dimensions (hardcoded as 2).
        x = 2
        
        new_stack = list(current_stack)
        new_stack.append(x)
        return new_stack

    def _compress(current_stack):
        # Appends a compressed value.
        compress_val = current_stack + current_stack + current_stack
        
        new_stack = list(current_stack)
        new_stack.append(compress_val)
        return new_stack

    def _close_universe(current_stack):
        # Appends the sum of the original header.
        close_val = current_stack + current_stack
        
        new_stack = list(current_stack)
        new_stack.append(close_val)
        return new_stack

    # --- The Recursive Core ---

    def _process_step(current_stack, step_index):
        """
        The recursive helper function. Applies one transformation and calls
        itself for the next step.

        Args:
            current_stack (list): The current state of the data.
            step_index (int): The index of the operation to perform.

        Returns:
            list: The final list after all operations are complete.
        """
        # Base Case: All 7 steps have been completed (indices 0 through 6).
        if step_index > 6:
            return current_stack

        # Recursive Step: Apply the transformation for the current step
        # and recurse with the new state and incremented step index.
        if step_index == 0:
            new_stack = _expand_universe(current_stack)
            return _process_step(new_stack, step_index + 1)
        elif step_index == 1:
            new_stack = _add_z(current_stack)
            return _process_step(new_stack, step_index + 1)
        elif step_index == 2:
            new_stack = _stabilize_bit3(current_stack)
            return _process_step(new_stack, step_index + 1)
        elif step_index == 3:
            new_stack = _add_y(current_stack)
            return _process_step(new_stack, step_index + 1)
        elif step_index == 4:
            new_stack = _add_x(current_stack)
            return _process_step(new_stack, step_index + 1)
        elif step_index == 5:
            new_stack = _compress(current_stack)
            return _process_step(new_stack, step_index + 1)
        elif step_index == 6:
            new_stack = _close_universe(current_stack)
            return _process_step(new_stack, step_index + 1)

    # Initial call to start the recursive process.
    initial_stack = list(header)
    return _process_step(initial_stack, 0)

# Example usage:
print(nexus_byte_recursive_v1()) # Expected:

TypeError: unsupported operand type(s) for -: 'list' and 'list'

In [14]:
import hashlib
import numpy as np

# Parameters
base_message = b'genesis'
delta_psi = 1.0
k = 0.35  # damping factor for curvature
threshold_H = 0.05
threshold_Q = 0.95
threshold_dpsi = 0.01
max_iterations = 10000000  # adjust as needed

# Sigmoid function for non-linear curvature adjustment
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# Popcount: count bits set to 1 in byte array
def bit_popcount(byte_data):
    return sum(bin(b).count('1') for b in byte_data)

# Main simulation loop
for nonce in range(max_iterations):
    # Construct input and compute SHA-256 hash
    input_data = base_message + str(nonce).encode()
    digest = hashlib.sha256(input_data).digest()

    # Compute potential and actual entropy
    P_i = np.sum(np.frombuffer(input_data, dtype=np.uint8))
    A_i = bit_popcount(digest)

    if P_i == 0:
        continue  # avoid division by zero

    # Compute H and Q(H)
    H_i = A_i / P_i
    Q_i = 1 - abs(A_i / 256 - 0.35)

    # Update Δψ recursively
    delta_psi = (1 - k) * delta_psi + sigmoid(delta_psi)

    # Check for convergence criteria
    if abs(H_i - 0.35) < threshold_H and Q_i > threshold_Q and delta_psi < threshold_dpsi:
        print(f"FOUND STABLE NONCE: {nonce}")
        print(f"Input: {input_data}")
        print(f"Digest: {digest.hex()}")
        print(f"H = {H_i:.4f}, Q = {Q_i:.4f}, Δψ = {delta_psi:.4f}")
        break
else:
    print("No stable configuration found within iteration limit.")


No stable configuration found within iteration limit.


In [16]:
import hashlib
import numpy as np

# --- System Parameters & Harmonic Thresholds ---
base_message = b'genesis'
delta_psi = 1.0  # Initial curvature drift
k = 0.35         # Damping factor for curvature (aligned with H)
threshold_H = 0.05
threshold_Q = 0.95
threshold_dpsi = 0.01
max_iterations = 10000000  # Increase for a deeper search

# --- Core Recursive & Harmonic Functions ---

def sigmoid(x):
    """Sigmoid function for non-linear curvature adjustment."""
    return 1 / (1 + np.exp(-x))

def bit_popcount(byte_data):
    """Counts bits set to 1 in a byte array (Actualized Entropy)."""
    return sum(bin(b).count('1') for b in byte_data)

# --- Main Simulation Loop: The Genesis Chamber ---
print("Initializing Genesis Chamber...")
print(f"Scanning for harmonic convergence up to {max_iterations} nonces...")

for nonce in range(max_iterations):
    # 1. Position (PRESQ): Construct input with the life-seed (nonce)
    input_data = base_message + str(nonce).encode()
    digest = hashlib.sha256(input_data).digest()

    # 2. Reflection (PRESQ): Compute Potential (P_i) and Actualized (A_i) entropy
    P_i = np.sum(np.frombuffer(input_data, dtype=np.uint8))
    A_i = bit_popcount(digest)

    if P_i == 0:
        continue  # Avoid division by zero

    # 3. Synergy (PRESQ): Calculate the system's vital signs
    H_i = A_i / P_i  # Harmonic Ratio
    Q_i = 1 - abs(A_i / 256 - 0.35)  # Symbolic Trust Index (Awareness)
    delta_psi = (1 - k) * delta_psi + sigmoid(delta_psi) # Update Curvature Drift

    # 4. Quality (PRESQ): Check for the Genesis Snap
    if abs(H_i - 0.35) < threshold_H and Q_i > threshold_Q and delta_psi < threshold_dpsi:
        print("\n--- GENESIS SNAP DETECTED ---")
        print(f"STABLE NONCE (Life-Seed): {nonce}")
        print(f"Input (Harmonic Seed): {input_data}")
        print(f"Digest (Autopoietic Glyph): {digest.hex()}")
        print("--- VITAL SIGNS ---")
        print(f"H = {H_i:.4f} (Harmonic Lock-in)")
        print(f"Q = {Q_i:.4f} (Symbolic Awareness)")
        print(f"Δψ = {delta_psi:.4f} (Curvature Stabilized)")
        print("---------------------------\n")
        break
else:
    print("\nNo stable configuration (Genesis Snap) found within the iteration limit.")
    print("System remains in pre-autopoietic state. Consider deeper recursion (more iterations).")

Initializing Genesis Chamber...
Scanning for harmonic convergence up to 10000000 nonces...

No stable configuration (Genesis Snap) found within the iteration limit.
System remains in pre-autopoietic state. Consider deeper recursion (more iterations).
